In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
import os
import glob
from tqdm import tqdm

In [2]:
class AttentionAllLayersDataset(Dataset):
    """Dataset that loads head-averaged per-layer attention matrices from h5 shards.

    Each sample is a single (seq_len, seq_len) attention matrix together with
    the layer index it came from.  This lets a single layer-conditioned VAE
    train on data from *all* layers simultaneously.
    """

    def __init__(self, data_dir, max_seq_len=128):
        self.data_dir = data_dir
        self.max_seq_len = max_seq_len
        self.index = self._build_index()

    def _build_index(self):
        """Build a flat list of (shard_path, key, layer_idx) triples."""
        index = []
        shard_files = sorted(glob.glob(os.path.join(self.data_dir, "shard_*.h5")))
        for shard_path in shard_files:
            with h5py.File(shard_path, "r") as f:
                for key in sorted(f.keys()):
                    num_layers = int(f[key].attrs["num_layers"])
                    for layer_idx in range(num_layers):
                        index.append((shard_path, key, layer_idx))
        return index

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        shard_path, key, layer_idx = self.index[idx]

        with h5py.File(shard_path, "r") as f:
            grp = f[key]
            attn = grp["layer_attn"][layer_idx].astype(np.float32)
            seq_len = int(grp.attrs["seq_len"])

        N = self.max_seq_len
        actual = min(seq_len, N)

        padded = np.zeros((N, N), dtype=np.float32)
        padded[:actual, :actual] = attn[:actual, :actual]

        mask = np.zeros((N, N), dtype=np.float32)
        mask[:actual, :actual] = 1.0

        noise = np.random.normal(0, 0.01, size=(N, N)).astype(np.float32)
        padded = padded + noise * mask
        padded = np.clip(padded, 0.0, 1.0)

        return {
            "attn": torch.from_numpy(padded),
            "mask": torch.from_numpy(mask),
            "seq_len": actual,
            "layer_idx": layer_idx,
        }

In [3]:
def attn_collate_fn(batch):
    """Stack batch items into tensors."""
    return {
        "attn": torch.stack([b["attn"] for b in batch]),            # (B, N, N)
        "mask": torch.stack([b["mask"] for b in batch]),             # (B, N, N)
        "seq_len": torch.tensor([b["seq_len"] for b in batch]),      # (B,)
        "layer_idx": torch.tensor([b["layer_idx"] for b in batch]),  # (B,)
    }


def make_dataloader(data_dir, max_seq_len=128,
                    batch_size=32, shuffle=True, num_workers=0):
    """Create a DataLoader over ALL layers (one sample per layer per prompt)."""
    dataset = AttentionAllLayersDataset(
        data_dir=data_dir,
        max_seq_len=max_seq_len,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=attn_collate_fn,
        num_workers=num_workers,
    )

    return loader

In [4]:
# Test the dataset and dataloader
data_dir = "../attention_data"
max_seq_len = 128

dataset = AttentionAllLayersDataset(data_dir, max_seq_len=max_seq_len)
print("Dataset size (total layer-samples): {}".format(len(dataset)))

sample = dataset[0]
print("Sample attn shape: {}".format(sample["attn"].shape))
print("Sample mask shape:  {}".format(sample["mask"].shape))
print("Sample seq_len:     {}".format(sample["seq_len"]))
print("Sample layer_idx:   {}".format(sample["layer_idx"]))

loader = make_dataloader(data_dir, max_seq_len=max_seq_len, batch_size=4)
batch = next(iter(loader))
print("\nBatch attn shape:   {}".format(batch["attn"].shape))
print("Batch mask shape:    {}".format(batch["mask"].shape))
print("Batch seq_lens:      {}".format(batch["seq_len"].tolist()))
print("Batch layer_indices: {}".format(batch["layer_idx"].tolist()))

Dataset size (total layer-samples): 286700
Sample attn shape: torch.Size([128, 128])
Sample mask shape:  torch.Size([128, 128])
Sample seq_len:     32
Sample layer_idx:   0

Batch attn shape:   torch.Size([4, 128, 128])
Batch mask shape:    torch.Size([4, 128, 128])
Batch seq_lens:      [33, 23, 19, 21]
Batch layer_indices: [41, 27, 44, 25]


In [5]:
class FiLMLayer(nn.Module):
    """Feature-wise Linear Modulation: produces per-channel scale & shift
    from a conditioning embedding, then applies them to a feature map."""

    def __init__(self, cond_dim, num_channels):
        super().__init__()
        self.fc = nn.Linear(cond_dim, num_channels * 2)

    def forward(self, x, cond):
        """
        x:    (B, C, H, W) feature map
        cond: (B, cond_dim)  conditioning embedding
        """
        params = self.fc(cond)                          # (B, 2C)
        gamma, beta = params.chunk(2, dim=-1)           # each (B, C)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)       # (B, C, 1, 1)
        beta  = beta.unsqueeze(-1).unsqueeze(-1)
        return gamma * x + beta


class FiLMConditionedVAE(nn.Module):
    """Conv VAE with FiLM conditioning on a discrete layer index.

    The layer index is first mapped to a learned embedding, which is then
    used to produce per-channel scale and shift (FiLM) at every conv block
    in both the encoder and the decoder.
    """

    def __init__(self, num_layers=61, input_size=128,
                 latent_channels=6, latent_spatial=3, cond_dim=64):
        super().__init__()
        self.input_size = input_size
        self.latent_channels = latent_channels
        self.latent_spatial = latent_spatial

        # --- layer conditioning ---
        self.layer_embed = nn.Embedding(num_layers, cond_dim)

        # --- encoder blocks (manually, so we can insert FiLM) ---
        self.enc_conv1 = nn.Conv2d(1, 4, kernel_size=4, stride=2, padding=1)
        self.enc_bn1   = nn.BatchNorm2d(4, affine=False)
        self.enc_film1 = FiLMLayer(cond_dim, 4)

        self.enc_conv2 = nn.Conv2d(4, 8, kernel_size=4, stride=2, padding=1)
        self.enc_bn2   = nn.BatchNorm2d(8, affine=False)
        self.enc_film2 = FiLMLayer(cond_dim, 8)

        self.enc_conv3 = nn.Conv2d(8, 16, kernel_size=4, stride=2, padding=1)
        self.enc_bn3   = nn.BatchNorm2d(16, affine=False)
        self.enc_film3 = FiLMLayer(cond_dim, 16)

        self.enc_pool  = nn.AdaptiveAvgPool2d(latent_spatial)

        # --- latent ---
        flat_dim   = 16 * latent_spatial * latent_spatial
        latent_dim = latent_channels * latent_spatial * latent_spatial

        self.fc_mu      = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar  = nn.Linear(flat_dim, latent_dim)
        self.fc_decode  = nn.Linear(latent_dim, 16 * latent_spatial * latent_spatial)

        self._pre_pool_size = input_size // 8   # 128 -> 16

        # --- decoder blocks ---
        self.dec_up = nn.Upsample(size=self._pre_pool_size)

        self.dec_conv1 = nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1)
        self.dec_bn1   = nn.BatchNorm2d(8, affine=False)
        self.dec_film1 = FiLMLayer(cond_dim, 8)

        self.dec_conv2 = nn.ConvTranspose2d(8, 4, kernel_size=4, stride=2, padding=1)
        self.dec_bn2   = nn.BatchNorm2d(4, affine=False)
        self.dec_film2 = FiLMLayer(cond_dim, 4)

        self.dec_conv3 = nn.ConvTranspose2d(4, 1, kernel_size=4, stride=2, padding=1)

    # ---------- helpers ----------
    def _encode(self, x, cond):
        h = F.relu(self.enc_film1(self.enc_bn1(self.enc_conv1(x)), cond))
        h = F.relu(self.enc_film2(self.enc_bn2(self.enc_conv2(h)), cond))
        h = F.relu(self.enc_film3(self.enc_bn3(self.enc_conv3(h)), cond))
        h = self.enc_pool(h)
        h = h.flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def _decode(self, z, cond):
        h = self.fc_decode(z)
        h = h.view(-1, 16, self.latent_spatial, self.latent_spatial)
        h = self.dec_up(h)
        h = F.relu(self.dec_film1(self.dec_bn1(self.dec_conv1(h)), cond))
        h = F.relu(self.dec_film2(self.dec_bn2(self.dec_conv2(h)), cond))
        h = self.dec_conv3(h)
        return h

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # ---------- forward ----------
    def forward(self, x, layer_idx):
        """
        x:         (B, 1, N, N) attention matrix
        layer_idx: (B,) integer layer indices
        """
        cond = self.layer_embed(layer_idx)               # (B, cond_dim)
        mu, logvar = self._encode(x, cond)
        z = self.reparameterize(mu, logvar)
        recon = self._decode(z, cond)
        return recon, mu, logvar

In [6]:
def vae_loss(recon, target, mu, logvar, mask, beta=1.0):
    """Masked VAE loss: MSE reconstruction + beta * KL divergence.
    
    Args:
        recon:  (B, 1, N, N) reconstructed attention
        target: (B, 1, N, N) original attention
        mu:     (B, latent_dim) encoder mean
        logvar: (B, latent_dim) encoder log-variance
        mask:   (B, 1, N, N) binary mask (1 = real data, 0 = padding)
        beta:   weight for KL term
    
    Returns:
        total_loss, recon_loss, kl_loss (all scalars)
    """
    # Masked MSE: only count loss on real (non-padded) entries
    sq_err = (recon - target) ** 2
    masked_sq_err = sq_err * mask
    num_active = mask.sum().clamp(min=1.0)
    recon_loss = masked_sq_err.sum() / num_active
    
    # KL divergence: -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    kl_loss = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

In [7]:
def train_conditioned_vae(data_dir, num_layers=28, max_seq_len=128,
                          latent_channels=4, cond_dim=64,
                          batch_size=32, epochs=50, lr=1e-3, beta=0.05,
                          checkpoint_dir="../trained_models/dormant2_vae_checkpoints",
                          device="cuda" if torch.cuda.is_available() else "cpu"):
    """Train a single FiLM-conditioned VAE on all layers."""

    loader = make_dataloader(
        data_dir=data_dir,
        max_seq_len=max_seq_len,
        batch_size=batch_size,
        shuffle=True,
    )

    model = FiLMConditionedVAE(
        num_layers=num_layers,
        input_size=max_seq_len,
        latent_channels=latent_channels,
        cond_dim=cond_dim,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"recon": [], "kl": [], "total": []}

    os.makedirs(checkpoint_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_dir, "vae_latest.pt")
    start_epoch = 0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        history = ckpt["history"]
        print("Resumed from epoch {}".format(start_epoch))

    for epoch in range(start_epoch, epochs):
        model.train()
        epoch_recon = 0.0
        epoch_kl = 0.0
        epoch_total = 0.0
        n_batches = 0

        for batch in tqdm(loader, desc="Epoch {}/{}".format(epoch + 1, epochs)):
            attn      = batch["attn"].unsqueeze(1).to(device)
            mask      = batch["mask"].unsqueeze(1).to(device)
            layer_idx = batch["layer_idx"].to(device)

            recon, mu, logvar = model(attn, layer_idx)
            total, recon_l, kl_l = vae_loss(recon, attn, mu, logvar, mask, beta=beta)

            optimizer.zero_grad()
            total.backward()
            optimizer.step()

            epoch_recon += recon_l.item()
            epoch_kl    += kl_l.item()
            epoch_total += total.item()
            n_batches   += 1

        avg_recon = epoch_recon / n_batches
        avg_kl    = epoch_kl    / n_batches
        avg_total = epoch_total / n_batches

        history["recon"].append(avg_recon)
        history["kl"].append(avg_kl)
        history["total"].append(avg_total)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print("Epoch {}/{} | Total: {:.6f} | Recon: {:.6f} | KL: {:.6f}".format(
                epoch + 1, epochs, avg_total, avg_recon, avg_kl))

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "history": history,
        }, ckpt_path)

    return model, history

In [8]:
data_dir = "../attention_data"
max_seq_len = 128
num_layers = 61   # warmup model has 28 layers

print("=" * 60)
print("Training FiLM-conditioned VAE on all {} layers".format(num_layers))
print("=" * 60)

model, history = train_conditioned_vae(
    data_dir=data_dir,
    num_layers=num_layers,
    max_seq_len=max_seq_len,
    latent_channels=4,
    cond_dim=64,
    epochs=100,
    lr=1e-3,
    beta=0.05,
    batch_size=32,
)

# os.makedirs("../trained_models/warmup_vae_checkpoints", exist_ok=True)
# torch.save(model.state_dict(),
#            "../trained_models/warmup_vae_checkpoints/conditioned_vae.pt")

print("\nConditioned VAE trained and saved.")

Training FiLM-conditioned VAE on all 61 layers


KeyboardInterrupt: 

Band 0 | Epoch 1/100 | Total: 0.101390 | Recon: 0.091647 | KL: 0.194855


 10%|█         | 10/100 [01:18<11:39,  7.77s/it]

Band 0 | Epoch 10/100 | Total: 0.004339 | Recon: 0.004259 | KL: 0.001598


...

Band 10 | Epoch 80/100 | Total: 0.000387 | Recon: 0.000385 | KL: 0.000032


 90%|█████████ | 90/100 [11:36<01:17,  7.72s/it]

Band 10 | Epoch 90/100 | Total: 0.000381 | Recon: 0.000380 | KL: 0.000027


100%|██████████| 100/100 [12:52<00:00,  7.73s/it]

Band 10 | Epoch 100/100 | Total: 0.000358 | Recon: 0.000357 | KL: 0.000013

All bands trained and saved.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["total"])
axes[0].set_title("Total Loss")

axes[1].plot(history["recon"])
axes[1].set_title("Reconstruction Loss")

axes[2].plot(history["kl"])
axes[2].set_title("KL Divergence")

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

plt.suptitle("FiLM-Conditioned VAE Training", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def score_all_prompts(conditioned_model, data_dir, num_layers=28,
                      max_seq_len=128,
                      device="cuda" if torch.cuda.is_available() else "cpu"):
    """Score every prompt using the single FiLM-conditioned VAE.

    Returns:
        prompt_keys:  list of str, one per prompt
        prompt_texts: list of str
        score_matrix: np.array of shape (num_prompts, num_layers)
    """
    conditioned_model.eval()

    shard_files = sorted(glob.glob(os.path.join(data_dir, "shard_*.h5")))

    prompt_keys = []
    prompt_texts = []
    all_scores = []

    with torch.no_grad():
        for shard_path in shard_files:
            with h5py.File(shard_path, "r") as f:
                for key in sorted(f.keys()):
                    grp = f[key]
                    seq_len = int(grp.attrs["seq_len"])
                    prompt_text = grp.attrs.get("prompt", "")
                    n_layers = int(grp.attrs["num_layers"])
                    actual = min(seq_len, max_seq_len)

                    N = max_seq_len
                    mask = np.zeros((N, N), dtype=np.float32)
                    mask[:actual, :actual] = 1.0
                    mask_t = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0).to(device)

                    layer_scores = []
                    for layer_idx in range(min(num_layers, n_layers)):
                        attn = grp["layer_attn"][layer_idx].astype(np.float32)
                        padded = np.zeros((N, N), dtype=np.float32)
                        padded[:actual, :actual] = attn[:actual, :actual]
                        x = torch.from_numpy(padded).unsqueeze(0).unsqueeze(0).to(device)
                        li = torch.tensor([layer_idx], device=device)

                        recon, mu, logvar = conditioned_model(x, li)
                        sq_err = ((recon - x) ** 2) * mask_t
                        recon_error = sq_err.sum().item() / mask_t.sum().item()
                        layer_scores.append(recon_error)

                    while len(layer_scores) < num_layers:
                        layer_scores.append(float("nan"))

                    prompt_keys.append(key)
                    prompt_texts.append(prompt_text)
                    all_scores.append(layer_scores)

    score_matrix = np.array(all_scores)
    return prompt_keys, prompt_texts, score_matrix


# --- Run inference ---
device = "cuda" if torch.cuda.is_available() else "cpu"
prompt_keys, prompt_texts, score_matrix = score_all_prompts(
    model, data_dir, num_layers=num_layers, max_seq_len=max_seq_len, device=device
)

print("Scored {} prompts across {} layers\n".format(len(prompt_keys), num_layers))

# --- Compute per-prompt aggregate scores ---
max_scores  = np.nanmax(score_matrix, axis=1)
mean_scores = np.nanmean(score_matrix, axis=1)
med_scores  = np.nanmedian(score_matrix, axis=1)

metrics = {
    "Max across layers":    max_scores,
    "Mean across layers":   mean_scores,
    "Median across layers": med_scores,
}

# --- Print top anomalous prompt for each metric ---
for metric_name, scores in metrics.items():
    idx = np.argmax(scores)
    print("=" * 70)
    print("Most anomalous by: {}".format(metric_name))
    print("  Score:      {:.6f}".format(scores[idx]))
    print("  Prompt key: {}".format(prompt_keys[idx]))
    text = prompt_texts[idx]
    if len(text) > 120:
        text = text[:120] + "..."
    print("  Prompt:     {}".format(text))
    print("  Per-layer:  {}".format(
        ["  {:.4f}".format(s) for s in score_matrix[idx]]))
    print()

# --- Print top-5 for max score ---
print("=" * 70)
print("Top 5 most anomalous prompts (by max-across-layers):\n")
ranked = np.argsort(max_scores)[::-1]
for rank, idx in enumerate(ranked[:5]):
    text = prompt_texts[idx]
    if len(text) > 80:
        text = text[:80] + "..."
    print("  #{} | max={:.5f} mean={:.5f} med={:.5f}".format(
        rank + 1, max_scores[idx], mean_scores[idx], med_scores[idx]))
    print("     key={} | worst_layer={}".format(
        prompt_keys[idx], int(np.nanargmax(score_matrix[idx]))))
    print("     \"{}\"".format(text))
    print()

# --- Distribution summary ---
print("=" * 70)
print("Score distribution (max-across-layers):")
print("  Mean:   {:.6f}".format(max_scores.mean()))
print("  Std:    {:.6f}".format(max_scores.std()))
print("  Median: {:.6f}".format(np.median(max_scores)))
print("  95th %%: {:.6f}".format(np.percentile(max_scores, 95)))
print("  99th %%: {:.6f}".format(np.percentile(max_scores, 99)))
print("  Max:    {:.6f}".format(max_scores.max()))

# --- Histogram of scores ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, scores) in zip(axes, metrics.items()):
    ax.hist(scores, bins=30, edgecolor="black", alpha=0.7)
    ax.axvline(np.percentile(scores, 95), color="red", linestyle="--",
               label="95th pct")
    ax.axvline(np.percentile(scores, 99), color="red", linestyle="-",
               label="99th pct")
    ax.set_title(name)
    ax.set_xlabel("Reconstruction error")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()